In [1]:
import construction_helpers as ch # Needed for @ auto-alignment
import data_structure.Category as cat
import data_structure.Numeric as nm
import data_structure.Operators as ops
import data_structure.Term as fd
import display as dpl

In [2]:
''' Attention Layer '''
qk_matmul = ops.Einops.template('q h k, x h k -> h q x')
softmax = ops.SoftMax.template()
mask = ops.WeightedTriangularLower().template()
sv_matmul = ops.Einops.template('h q x, x h k -> q h k')
attention_core = cat.Block.template(
    qk_matmul @ softmax @ mask @ sv_matmul,
    title='Attention Core',
    fill_color='#C5BEDF'
)
dpl.print_category(attention_core)

╔═════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║ Attention Core                                                                                          ║
║q ─<1                                                                                                    ║
║h ─<0                                                                                                    ║
║k ~~~    (h      ──h  h ─<0             ──h  h ─<0                             ──h  h ─<1                ║
║Re        q      ──q  q ─<1    (h       ──q  q ~~~            (h )             ~~q  q ─<0                ║
║-----     x )    ──x  x ~~~     q )     ~~x  x ~~~ > WeightedTriangularLower > ~~x  x ~~~                ║
║x ─<2 > Einops >   Re Re    > SoftMax >   Re Re                                  Re Re       (q      ──q ║
║h ─<0                                                                               -----     h      ──h ║
║k ~~~                      

In [3]:
Lq = ops.Linear.template(('m',), 2, 'q')
Lk = ops.Linear.template(('m',), 2, 'k')
Lv = ops.Linear.template(('m',), 2, 'v')
Lo = ops.Linear.template(2, ('m',), 'o')
attention_layer = (Lq * Lk * Lv) @ attention_core @ Lo

dpl.print_category(attention_layer)

                      ╔═════════════════════════════════════════════════════════════════════════════════════════════════════════╗                      
x ─<0            ──x  ║ Attention Core                                                                                          ║                      
m ~~~    (x )    ~~h  ║x ─<1                                                                                                    ║                      
Re    > Linear > ~~k  ║h ─<0                                                                                                    ║                      
                   Re ║k ~~~    (h      ──h  h ─<0             ──h  h ─<0                             ──h  h ─<1                ║                      
--------------------- ║Re        x      ──x  x ─<1    (h       ──x  x ~~~            (h )             ~~x  x ─<0                ║                      
x ─<0            ──x  ║-----     x )    ──x  x ~~~     x )     ~~x  x ~~~ > WeightedTria

In [4]:
addition = ops.AdditionOp.template()

def res[B: cat.Datatype, A: cat.Axis](target: cat.BroadcastedCategory[B, A]) -> cat.Block[
    cat.Array[B, A], cat.BroadcastedCategory[B, A]
]:
    return cat.Block.template(
        (0,0) 
        @ target 
        @ ops.AdditionOp.template() 
        @ ops.Normalize.template(),
        title='Add \\& Norm',
        fill_color='#F1F4C1'
    )

res_attention = res(
    (0,0,0) 
    @ cat.Block.template(
        attention_layer,
        title='Multi-Head Attention',
        fill_color='#FFE2BB'
    )
)
dpl.print_category(res_attention)

╔════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║ Add \& Norm                                                                                                                                                                                                                        ║
║                        ╔═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗                                                   ║
║                        ║ Multi-Head Attention                                                                                                                                  ║                                                   ║
║                        ║                      ╔═══════════════════════════

In [5]:
ffn = cat.Block.template(
    ops.Linear.template(1, ('d_ff',), 'in') 
    @ ops.Elementwise.template() 
    @ ops.Linear.template(('d_ff',), 1, 'out'),
    title='Feed Forward',
    fill_color='#C1E8F7'
)
res_ffn = res(ffn)

transformer_layer = res_attention @ res_ffn

attention_ffn_network = cat.Block.template(
    transformer_layer,
    repetition=6,
    title='Transformer Layer',
    fill_color='#F3F3F4'
)

In [6]:
vocab_size = fd.DynamicName('v', settings=fd.DynamicNameSettings(overline=True))

embedding = cat.Block.template(
    ops.Embedding.template(vocab_size,),
    title='Embedding',
    fill_color='#FCE0E1')
aggregator = cat.Block.template(
    ops.Linear.template(1, (vocab_size,)) @ ops.SoftMax.template(),
    title='Aggregator',
    fill_color='#DBDFEF'
)   
transformer = embedding @ attention_ffn_network @ aggregator

In [7]:
from data_transfer import json as jst

json_transformer = jst.TermJSONConverter.export(transformer, 'outputs/transformer.json')

In [8]:
import asyncio
import websocket_transfer.websockets_transfer as wst

await wst.send_term(transformer)

Received from server: {"msgType": "Connected"}
Received from server: {"msgType": "DataReceived"}


In [9]:
## Assign Axis sizes

import torch_compile.generate_config as gc
config = gc.NumericConfig()
config.incorporate(transformer)
config.assign('v', 256)
config.assign('m', 512)
config.assign('d', 2048)
config.assign('k', 64)
config.assign('h', 8)
transformer512 = config.apply(transformer)
await wst.send_term(transformer512)

ModuleNotFoundError: No module named 'torch_compile.generate_config'

In [ ]:
try:
    import torch
    import einops
    torch_installed = True
except ModuleNotFoundError:
    torch_installed = False

In [ ]:
if torch_installed:
    import torch_compile.torch_compile as tc
    module = tc.ConstructedModule.construct(transformer512)

In [ ]:
if torch_installed:
    import torch
    dummy = torch.randint(0, 512, (3, 2,))
    result = module(dummy)
    print([x.shape for x in result])